# PINN advantage #2: a fourth-order PDE (the Euler–Bernoulli beam)

Solve the static deflection of a **simply-supported Euler–Bernoulli beam** under a uniform load, a fourth-order ODE:
$$\frac{\mathrm{d}^4 u}{\mathrm{d}x^4}(x) = q,\qquad x\in[0,1],\qquad u(0)=u(1)=0,\quad u''(0)=u''(1)=0.$$
Here $EI=1$ and the load is uniform, $q=1$. The two conditions at each end are the **displacement** ($u=0$) and the **bending moment** ($u''=0$).

**Exact solution.** $\;u^\star(x)=\dfrac{q}{24}\,(x^4-2x^3+x)$, with mid-span deflection $u^\star(\tfrac12)=\dfrac{5q}{384}=0.01302$.

**Why a grid solver struggles.** A finite-difference treatment of a fourth derivative needs a wide (5-point) stencil, and imposing *two* boundary conditions at each end — one on $u$, one on $u''$ — requires ghost nodes and careful bookkeeping. A PINN needs neither: the fourth derivative comes from nesting automatic differentiation four times, and it is **exact**.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Q0 = 1.0                                  # uniform load (EI = 1)
def u_exact(x): return (Q0/24.0)*(x**4 - 2*x**3 + x)

def D(f, x):                              # one derivative, graph kept alive
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

The trial solution $u(x)=x(1-x)\,\mathcal N(x)$ builds the two **displacement** conditions $u(0)=u(1)=0$ in exactly; only the **moment** conditions $u''(0)=u''(1)=0$ are left to the loss. The interior residual is $u''''-q$, with the fourth derivative obtained by four nested calls to `D`.

In [ ]:
net = nn.Sequential(nn.Linear(1,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
opt = torch.optim.Adam(net.parameters(), 3e-3)

def U(x):                                 # hard-enforces u(0)=u(1)=0
    return x*(1.0 - x)*net(x)

x0 = torch.zeros(1,1, device=device, requires_grad=True)
x1 = torch.ones (1,1, device=device, requires_grad=True)

t0 = time.perf_counter()
for e in range(6000):
    if e == 4500:
        for g in opt.param_groups: g['lr'] = 5e-4
    opt.zero_grad()
    x = torch.rand(1000,1, device=device).requires_grad_(True)
    u = U(x)
    uxxxx = D(D(D(D(u, x), x), x), x)      # exact 4th derivative
    loss = ((uxxxx - Q0)**2).mean()
    for xe in (x0, x1):                    # simply-supported moment BC  u''=0
        ue = U(xe); loss = loss + 20.0*(D(D(ue, xe), xe)**2).mean()
    loss.backward(); opt.step()
    if e % 1000 == 0:
        print(f'epoch {e:5d}   loss {loss.item():.2e}')
if device.type == 'cuda': torch.cuda.synchronize()
train_time = time.perf_counter() - t0
print(f'\ntraining time: {train_time:.1f} s')

In [ ]:
xg = np.linspace(0, 1, 201)
u_ex = u_exact(xg)
with torch.no_grad():
    xt = torch.tensor(xg, dtype=torch.float32, device=device).reshape(-1,1)
    u_pn = (xt*(1.0 - xt)*net(xt)).cpu().numpy().ravel()
rel = np.sqrt(np.mean((u_pn - u_ex)**2)/np.mean(u_ex**2))
print(f'relative L2 error : {rel:.2e}')
print(f'mid-span deflection: exact {u_ex.max():.5f}   PINN {u_pn.max():.5f}   (5q/384 = {5*Q0/384:.5f})')

plt.figure(figsize=(7.5,4.2))
plt.plot(xg, u_ex, 'g', lw=2.8, alpha=.6, label='exact')
plt.plot(xg, u_pn, 'r--', lw=1.7, label=f'PINN (rel L2 = {rel:.1e})')
plt.xlabel('x'); plt.ylabel('deflection u(x)'); plt.legend(loc='lower center')
plt.grid(alpha=.3); plt.gca().invert_yaxis()
plt.title("Fourth-order PDE: simply-supported beam  u'''' = q")
plt.tight_layout(); plt.show()

The network matches the analytic deflection to a relative $L_2$ error of a few $\times10^{-5}$, recovering the textbook mid-span value $5q/384$. Nothing about the method changed from a second-order problem except the number of times `D` is nested — the fourth derivative is read off the computational graph exactly, with no stencil and no ghost nodes.